# Traffic Demand Prediction — Flipkart Gridlock Hackathon 2.0
This notebook contains the complete, high-accuracy traffic demand prediction pipeline using an ensemble of **LightGBM**, **XGBoost**, and **CatBoost** regressors trained on advanced spatio-temporal features.

### Key Highlights:
1. **Zero Validation Leakage**: Encodings and target mappings are calculated strictly inside each validation fold (fold-safe).
2. **Geohash Alignment & Collapse**: Geohash string indices are preserved for structural mappings, and rare categories (frequency < 5) are collapsed into `'OTHER'` to prevent overfitting.
3. **Spatial Clustering**: Decoded coordinates are clustered via KMeans to define geographic zones.
4. **Advanced Historical Lags & Rolling Statistics**: Lags and rolling window statistics are computed on Day 48 demand series.
5. **Interaction Encodings**: Strong interaction features are engineered (e.g. `RoadType x Hour`, `Weather x Hour`, `lane_hour`).
6. **Weighted Spatio-Temporal CV Ensemble**: Robust out-of-fold bagging combining LightGBM, XGBoost, and CatBoost.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import pygeohash as pgh
from lightgbm import LGBMRegressor, early_stopping
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder
import os
import warnings
warnings.filterwarnings('ignore')

## 2. Load Datasets

In [ ]:
train = pd.read_csv("dataset/train.csv")
test = pd.read_csv("dataset/test.csv")

train['geohash_raw'] = train['geohash'].astype(str)
test['geohash_raw'] = test['geohash'].astype(str)

## 3. Preprocessing & Temporal Features

In [ ]:
def process_time(df):
    df['dt'] = pd.to_datetime(df['timestamp'], format='%H:%M')
    df['hour'] = df['dt'].dt.hour
    df['minute'] = df['dt'].dt.minute
    df['time_slot'] = df['hour'] * 4 + df['minute'] // 15
    df['time_slot_sin'] = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['time_slot_cos'] = np.cos(2 * np.pi * df['time_slot'] / 96)
    
    # Weekday and weekend features (day is sequential)
    df['day_of_week'] = df['day'] % 7
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    # Peak hour, night, and office hour intelligence
    df['is_peak_hour'] = (((df['hour'] >= 7) & (df['hour'] <= 10)) | ((df['hour'] >= 17) & (df['hour'] <= 20))).astype(int)
    df['is_night'] = ((df['hour'] >= 22) | (df['hour'] < 6)).astype(int)
    df['is_office_hour'] = ((df['hour'] >= 9) & (df['hour'] < 17) & (df['is_weekend'] == 0)).astype(int)
    
    df.drop(columns=['dt'], inplace=True)
    return df

train = process_time(train)
test = process_time(test)

## 4. Spatiotemporal Coordinates & Clustering

In [ ]:
print("Decoding geohashes to lat/lon...")
geohashes = pd.concat([train['geohash_raw'], test['geohash_raw']]).unique()
geo_coords = {}
for g in geohashes:
    try:
        lat, lon = pgh.decode(g)
        geo_coords[g] = (lat, lon)
    except Exception:
        geo_coords[g] = (np.nan, np.nan)

train['lat'] = train['geohash_raw'].map(lambda g: geo_coords[g][0])
train['lon'] = train['geohash_raw'].map(lambda g: geo_coords[g][1])
test['lat'] = test['geohash_raw'].map(lambda g: geo_coords[g][0])
test['lon'] = test['geohash_raw'].map(lambda g: geo_coords[g][1])

train['lat'] = train['lat'].fillna(train['lat'].mean())
train['lon'] = train['lon'].fillna(train['lon'].mean())
test['lat'] = test['lat'].fillna(train['lat'].mean())
test['lon'] = test['lon'].fillna(train['lon'].mean())

# Spatial Clustering (KMeans on lat/lon)
print("Fitting spatial clustering...")
kmeans = KMeans(n_clusters=10, random_state=42, n_init='auto')
kmeans.fit(train[['lat', 'lon']])
train['spatial_cluster'] = kmeans.labels_
test['spatial_cluster'] = kmeans.predict(test[['lat', 'lon']])

## 5. Category Collapsing & Value Imputation

In [ ]:
print("Collapsing rare geohashes...")
geohash_counts = train['geohash_raw'].value_counts()
rare_geohashes = set(geohash_counts[geohash_counts < 5].index)
train['geohash_raw'] = train['geohash_raw'].apply(lambda g: 'OTHER' if g in rare_geohashes else g)
test['geohash_raw'] = test['geohash_raw'].apply(lambda g: 'OTHER' if g in rare_geohashes else g)

print("Imputing missing values...")
temp_median = train['Temperature'].median()
geo_temp_median = train.groupby('geohash_raw')['Temperature'].median().to_dict()

def impute_temp(row):
    if pd.isna(row['Temperature']):
        return geo_temp_median.get(row['geohash_raw'], temp_median)
    return row['Temperature']

train['Temperature'] = train.apply(impute_temp, axis=1)
test['Temperature'] = test.apply(impute_temp, axis=1)

train['Weather'] = train['Weather'].fillna("Unknown")
test['Weather'] = test['Weather'].fillna("Unknown")
train['RoadType'] = train['RoadType'].fillna("Unknown")
test['RoadType'] = test['RoadType'].fillna("Unknown")

## 6. Interaction Features

In [ ]:
train['geo_4'] = train['geohash_raw'].str[:4]
train['geo_5'] = train['geohash_raw'].str[:5]
test['geo_4'] = test['geohash_raw'].str[:4]
test['geo_5'] = test['geohash_raw'].str[:5]

def create_interactions(df):
    df['road_hour'] = df['RoadType'].astype(str) + "_" + df['hour'].astype(str)
    df['weather_hour'] = df['Weather'].astype(str) + "_" + df['hour'].astype(str)
    df['lane_hour'] = df['NumberofLanes'].astype(str) + "_" + df['hour'].astype(str)
    df['geo_hour'] = df['geohash_raw'] + "_" + df['hour'].astype(str)
    return df

train = create_interactions(train)
test = create_interactions(test)

print("Encoding categoricals...")
cat_cols = ['Weather', 'RoadType', 'LargeVehicles', 'Landmarks', 'geohash', 'geo_4', 'geo_5', 'road_hour', 'weather_hour', 'lane_hour', 'geo_hour']
for col in cat_cols:
    le = LabelEncoder()
    if col == 'geohash':
        combined = pd.concat([train['geohash_raw'].astype(str), test['geohash_raw'].astype(str)])
        le.fit(combined)
        train['geohash_encoded'] = le.transform(train['geohash_raw'].astype(str))
        test['geohash_encoded'] = le.transform(test['geohash_raw'].astype(str))
    else:
        combined = pd.concat([train[col].astype(str), test[col].astype(str)])
        le.fit(combined)
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))

## 7. Lags and Rolling Statistics on Day 48

In [ ]:
train_48 = train[train['day'] == 48].copy()
train_49 = train[train['day'] == 49].copy()

d48_demand_dict = train_48.set_index(['geohash_raw', 'time_slot'])['demand'].to_dict()

def get_d48_demand(geohash, slot, offset=0):
    t_slot = slot + offset
    if t_slot < 0 or t_slot > 95:
        return np.nan
    return d48_demand_dict.get((geohash, t_slot), np.nan)

for df in [train_49, test]:
    df['demand_prev_day'] = df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], 0), axis=1)
    df['lag_1_d48'] = df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], -1), axis=1)
    df['lag_2_d48'] = df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], -2), axis=1)
    df['lag_3_d48'] = df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], -3), axis=1)
    df['lag_6_d48'] = df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], -6), axis=1)
    df['lag_12_d48'] = df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], -12), axis=1)
    df['lag_24_d48'] = df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], -24), axis=1)
    
    slot_mean_d48 = train_48.groupby('time_slot')['demand'].mean().to_dict()
    df['slot_mean_d48'] = df['time_slot'].map(slot_mean_d48)
    
    for col in ['demand_prev_day', 'lag_1_d48', 'lag_2_d48', 'lag_3_d48', 'lag_6_d48', 'lag_12_d48', 'lag_24_d48']:
        df[col] = df[col].fillna(df['slot_mean_d48'])
        
    df['rolling_mean_3_d48'] = (df['lag_1_d48'] + df['demand_prev_day'] + df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], 1), axis=1).fillna(df['slot_mean_d48'])) / 3.0
    df['rolling_mean_6_d48'] = (df['lag_2_d48'] + df['lag_1_d48'] + df['demand_prev_day'] + 
                                df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], 1), axis=1).fillna(df['slot_mean_d48']) + 
                                df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], 2), axis=1).fillna(df['slot_mean_d48']) + 
                                df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], 3), axis=1).fillna(df['slot_mean_d48'])) / 6.0
    df['rolling_mean_12_d48'] = (df['lag_6_d48'] + df['lag_3_d48'] + df['lag_2_d48'] + df['lag_1_d48'] + df['demand_prev_day'] + 
                                 df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], 1), axis=1).fillna(df['slot_mean_d48']) + 
                                 df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], 2), axis=1).fillna(df['slot_mean_d48']) + 
                                 df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], 3), axis=1).fillna(df['slot_mean_d48']) +
                                 df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], 4), axis=1).fillna(df['slot_mean_d48']) +
                                 df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], 5), axis=1).fillna(df['slot_mean_d48']) +
                                 df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], 6), axis=1).fillna(df['slot_mean_d48'])) / 11.0
                                 
    df['rolling_std_3_d48'] = np.std([df['lag_1_d48'], df['demand_prev_day'], df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], 1), axis=1).fillna(df['slot_mean_d48'])], axis=0)
    df['rolling_max_3_d48'] = np.max([df['lag_1_d48'], df['demand_prev_day'], df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], 1), axis=1).fillna(df['slot_mean_d48'])], axis=0)
    df['rolling_min_3_d48'] = np.min([df['lag_1_d48'], df['demand_prev_day'], df.apply(lambda r: get_d48_demand(r['geohash_raw'], r['time_slot'], 1), axis=1).fillna(df['slot_mean_d48'])], axis=0)

train_48['demand_prev_day'] = np.nan
train_48['lag_1_d48'] = np.nan
train_48['lag_2_d48'] = np.nan
train_48['lag_3_d48'] = np.nan
train_48['lag_6_d48'] = np.nan
train_48['lag_12_d48'] = np.nan
train_48['lag_24_d48'] = np.nan
train_48['rolling_mean_3_d48'] = np.nan
train_48['rolling_mean_6_d48'] = np.nan
train_48['rolling_mean_12_d48'] = np.nan
train_48['rolling_std_3_d48'] = np.nan
train_48['rolling_max_3_d48'] = np.nan
train_48['rolling_min_3_d48'] = np.nan

# Statistical Aggregates on Day 48
geo_mean_d48 = train_48.groupby('geohash_raw')['demand'].mean().to_dict()
geo_std_d48 = train_48.groupby('geohash_raw')['demand'].std().to_dict()
geo_hour_mean_d48 = train_48.groupby('geo_hour')['demand'].mean().to_dict()
hour_mean_d48 = train_48.groupby('hour')['demand'].mean().to_dict()
weather_mean_d48 = train_48.groupby('Weather')['demand'].mean().to_dict()
slot_mean_d48 = train_48.groupby('time_slot')['demand'].mean().to_dict()

for df in [train_48, train_49, test]:
    df['geo_mean_d48'] = df['geohash_raw'].map(geo_mean_d48).fillna(df['time_slot'].map(slot_mean_d48))
    df['geo_std_d48'] = df['geohash_raw'].map(geo_std_d48).fillna(0.0)
    df['geo_hour_mean_d48'] = df['geo_hour'].map(geo_hour_mean_d48).fillna(df['geo_mean_d48'])
    df['hour_mean_d48'] = df['hour'].map(hour_mean_d48).fillna(df['time_slot'].map(slot_mean_d48))
    df['weather_mean_d48'] = df['Weather'].map(weather_mean_d48).fillna(df['time_slot'].map(slot_mean_d48))
    df['slot_mean_d48'] = df['time_slot'].map(slot_mean_d48)

## 8. Frequency Encodings

In [ ]:
print("Frequency encoding categoricals...")
combined_train = pd.concat([train_48, train_49], ignore_index=True)
freq_cols = ['geohash_raw', 'geo_hour', 'road_hour']
for col in freq_cols:
    freq_map = combined_train[col].value_counts().to_dict()
    train_48[f'{col}_freq'] = train_48[col].map(freq_map)
    train_49[f'{col}_freq'] = train_49[col].map(freq_map)
    test[f'{col}_freq'] = test[col].map(freq_map).fillna(0.0)

## 9. Define Features space

In [ ]:
features = [
    'geohash_encoded', 'geo_4', 'geo_5', 'time_slot_sin', 'time_slot_cos',
    'lat', 'lon', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks',
    'Temperature', 'Weather', 'demand_prev_day',
    'lag_1_d48', 'lag_2_d48', 'lag_3_d48', 'lag_6_d48', 'lag_12_d48', 'lag_24_d48',
    'rolling_mean_3_d48', 'rolling_mean_6_d48', 'rolling_mean_12_d48', 'rolling_std_3_d48', 'rolling_max_3_d48', 'rolling_min_3_d48',
    'geo_mean_d48', 'geo_std_d48', 'geo_hour_mean_d48', 'hour_mean_d48', 'weather_mean_d48', 'slot_mean_d48',
    'road_hour', 'weather_hour', 'lane_hour', 'geo_hour',
    'geohash_raw_freq', 'geo_hour_freq', 'road_hour_freq', 'is_weekend', 'is_peak_hour', 'is_night', 'is_office_hour', 'spatial_cluster'
]

X = train_49[features]
y = train_49['demand']
X_test = test[features]

## 10. CV Training with Fold-Safe Target Encoding

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_49))
test_preds = np.zeros(len(test))

te_cols = ['geohash_encoded', 'geo_hour', 'road_hour']

for fold, (train_idx, val_idx) in enumerate(kf.split(train_49)):
    print(f"--- FOLD {fold+1} ---")
    train_49_fold = train_49.iloc[train_idx].copy()
    val_fold = train_49.iloc[val_idx].copy()
    
    combined_train_fold = pd.concat([train_48, train_49_fold], ignore_index=True)
    
    X_tr = combined_train_fold[features].copy()
    y_tr = combined_train_fold['demand'].copy()
    X_va = val_fold[features].copy()
    y_va = val_fold['demand'].copy()
    X_te = test[features].copy()
    
    # Fold-safe target encoding computed strictly on combined_train_fold
    for col in te_cols:
        target_mean = y_tr.groupby(X_tr[col]).mean().to_dict()
        global_mean = y_tr.mean()
        
        X_tr[f'{col}_te'] = X_tr[col].map(target_mean).fillna(global_mean)
        X_va[f'{col}_te'] = X_va[col].map(target_mean).fillna(global_mean)
        X_te[f'{col}_te'] = X_te[col].map(target_mean).fillna(global_mean)
        
    X_tr_fit = X_tr.drop(columns=te_cols)
    X_va_fit = X_va.drop(columns=te_cols)
    X_te_fit = X_te.drop(columns=te_cols)
    
    # LightGBM
    lgb = LGBMRegressor(n_estimators=3000, learning_rate=0.03, num_leaves=63, min_child_samples=50, feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1, lambda_l1=0.1, lambda_l2=0.1, random_state=42, verbose=-1)
    lgb.fit(X_tr_fit, y_tr, eval_set=[(X_va_fit, y_va)], callbacks=[early_stopping(stopping_rounds=100, verbose=False)])
    pred_lgb = lgb.predict(X_va_fit)
    
    # XGBoost
    xgb = XGBRegressor(n_estimators=3000, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, early_stopping_rounds=100, random_state=42, verbosity=0)
    xgb.fit(X_tr_fit, y_tr, eval_set=[(X_va_fit, y_va)], verbose=False)
    pred_xgb = xgb.predict(X_va_fit)
    
    # CatBoost
    cat = CatBoostRegressor(iterations=3000, learning_rate=0.03, depth=6, subsample=0.8, early_stopping_rounds=100, random_state=42, verbose=0)
    cat.fit(X_tr_fit, y_tr, eval_set=[(X_va_fit, y_va)], verbose=False)
    pred_cat = cat.predict(X_va_fit)
    
    pred_ens = 0.5 * pred_lgb + 0.35 * pred_xgb + 0.15 * pred_cat
    oof_preds[val_idx] = pred_ens
    
    test_preds += (0.5 * lgb.predict(X_te_fit) + 0.35 * xgb.predict(X_te_fit) + 0.15 * cat.predict(X_te_fit)) / 5.0
    
    print(f"Fold {fold+1} Ensemble R2: {r2_score(y_va, pred_ens):.5f}")

print(f"Overall Ensemble OOF R2: {r2_score(y, oof_preds):.5f}")

## 11. Generate Submission

In [ ]:
test_preds_clipped = np.clip(test_preds, 0.0, 1.0)
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': test_preds_clipped
})
submission.to_csv("submission.csv", index=False)
print("Submission saved successfully!")